# Домашняя работа по эмпирическим отраслевым рынкам

Ноутбук использует:
- `repair_shops.csv`
- `homework_industries.xlsx` (лист `Данные`)

Все таблицы и графики сохраняются в `/mnt/data/hw_results`.

## 1. Подготовка: импорт библиотек и пути к файлам

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from scipy.optimize import minimize
from scipy.stats import norm

# =========================
# Settings
# =========================
REPAIR_FILE = "/mnt/data/repair_shops.csv"
INDUSTRY_FILE = "/mnt/data/homework_industries.xlsx"

# Put your sheet name here. If None or not found, the first sheet is used.
SHEET_NAME = None

OUTPUT_DIR = "/mnt/data/hw_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def save_df(df: pd.DataFrame, filename: str) -> str:
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=True, encoding="utf-8-sig")
    return path


def result_to_table(result, index_name: str = "term") -> pd.DataFrame:
    """Turn a statsmodels result into a clean coefficient table."""
    table = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "t_or_z": result.tvalues,
        "p_value": result.pvalues,
    })
    ci = result.conf_int()
    ci.columns = ["ci_low", "ci_high"]
    table = pd.concat([table, ci], axis=1)
    table.index.name = index_name
    return table


def safe_log_df(df: pd.DataFrame, cols):
    """Keep only rows where all selected columns are strictly positive and log-transform them."""
    work = df.loc[:, cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    work = work[(work > 0).all(axis=1)].copy()
    log_work = np.log(work)
    return work, log_work

## 2. Вспомогательные функции

In [ ]:
def save_df(df: pd.DataFrame, filename: str) -> str:
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=True, encoding="utf-8-sig")
    return path


def result_to_table(result, index_name: str = "term") -> pd.DataFrame:
    """Turn a statsmodels result into a clean coefficient table."""
    table = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "t_or_z": result.tvalues,
        "p_value": result.pvalues,
    })
    ci = result.conf_int()
    ci.columns = ["ci_low", "ci_high"]
    table = pd.concat([table, ci], axis=1)
    table.index.name = index_name
    return table


def safe_log_df(df: pd.DataFrame, cols):
    """Keep only rows where all selected columns are strictly positive and log-transform them."""
    work = df.loc[:, cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    work = work[(work > 0).all(axis=1)].copy()
    log_work = np.log(work)
    return work, log_work


## 3. Часть 1: анализ `repair_shops.csv`

In [ ]:
repair = pd.read_csv(REPAIR_FILE)
repair.columns = repair.columns.str.lower()

# Rename "housing" to a more readable market-size label used in the assignment.
repair = repair.rename(columns={"housing": "houseunits"})

# 1) Descriptive statistics
repair_desc = repair[[
    "placeid",
    "houseunits",
    "population",
    "autoroute",
    "est811111",
    "est811112",
    "est811113",
]].describe().T
save_df(repair_desc, "repair_descriptive_statistics.csv")

print("\n=== Repair shops: descriptive statistics ===")
print(repair_desc)

# Distribution of workshop counts across market-size quartiles
repair["pop_quartile"] = pd.qcut(
    repair["population"],
    q=4,
    labels=["Q1_smallest", "Q2", "Q3", "Q4_largest"]
)

avg_by_quartile = repair.groupby("pop_quartile", observed=False)[
    ["est811111", "est811112", "est811113"]
].mean()

save_df(avg_by_quartile, "repair_average_workshops_by_population_quartile.csv")

# Plot workshop-count distributions overall
for col in ["est811111", "est811112", "est811113"]:
    fig, ax = plt.subplots(figsize=(8, 4))
    max_val = int(repair[col].max())
    bins = np.arange(-0.5, max_val + 1.5, 1)
    ax.hist(repair[col], bins=bins)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel("Number of workshops")
    ax.set_ylabel("Number of cities")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"{col}_distribution.png"), dpi=200)
    plt.close(fig)

# Plot how average workshop counts vary by city size
fig, ax = plt.subplots(figsize=(9, 5))
avg_by_quartile.plot(kind="bar", ax=ax)
ax.set_title("Average number of workshops by population quartile")
ax.set_xlabel("Population quartile")
ax.set_ylabel("Average number of workshops")
ax.legend(title="Workshop type")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "workshops_by_population_quartile.png"), dpi=200)
plt.close(fig)

# 2) Two OLS regressions:
#    (a) transmission shops on population
#    (b) exhaust shops on population
X_pop = sm.add_constant(repair["population"])

ols_trans = sm.OLS(repair["est811112"], X_pop).fit(cov_type="HC1")
ols_exhaust = sm.OLS(repair["est811113"], X_pop).fit(cov_type="HC1")

trans_table = result_to_table(ols_trans)
exhaust_table = result_to_table(ols_exhaust)

save_df(trans_table, "repair_ols_transmission.csv")
save_df(exhaust_table, "repair_ols_exhaust.csv")

resid_corr = np.corrcoef(ols_trans.resid, ols_exhaust.resid)[0, 1]
with open(os.path.join(OUTPUT_DIR, "repair_residual_correlation.txt"), "w", encoding="utf-8") as f:
    f.write(f"Correlation of residuals: {resid_corr:.6f}\n")

print("\n=== OLS: transmission shops on population ===")
print(ols_trans.summary())
print("\n=== OLS: exhaust shops on population ===")
print(ols_exhaust.summary())
print(f"\nResidual correlation: {resid_corr:.6f}")

# 3) Ordered probit for transmission shops
#    We turn the count variable into ordered categories based on the observed distribution.
#    0 shops, 1 shop, 2 shops, 3+ shops.
repair["trans_cat"] = np.select(
    [
        repair["est811112"] == 0,
        repair["est811112"] == 1,
        repair["est811112"] == 2,
        repair["est811112"] >= 3,
    ],
    [0, 1, 2, 3],
    default=np.nan,
).astype(int)

repair["trans_cat"] = pd.Categorical(repair["trans_cat"], categories=[0, 1, 2, 3], ordered=True)

exog_op = pd.DataFrame({
    "log_population": np.log(repair["population"]),
    "autoroute": repair["autoroute"].astype(float),
})

op_model = OrderedModel(repair["trans_cat"], exog_op, distr="probit")
op_res = op_model.fit(method="bfgs", disp=False)

op_table = pd.DataFrame({
    "coef": op_res.params,
    "std_err": op_res.bse,
    "z": op_res.tvalues,
    "p_value": op_res.pvalues,
})
save_df(op_table, "repair_ordered_probit.csv")

print("\n=== Ordered probit: transmission-shop categories ===")
print(op_res.summary())

## 4. Часть 2: анализ `homework_industries.xlsx`

In [ ]:
xl = pd.ExcelFile(INDUSTRY_FILE)
print("\nAvailable sheets:")
for s in xl.sheet_names:
    print(" -", s)

sheet_name = SHEET_NAME if (SHEET_NAME in xl.sheet_names) else xl.sheet_names[0]
print(f"\nUsing sheet: {sheet_name}")

industry = pd.read_excel(INDUSTRY_FILE, sheet_name=sheet_name)

industry_desc = industry.describe().T
save_df(industry_desc, "industry_descriptive_statistics.csv")

print("\n=== Industry data: descriptive statistics ===")
print(industry_desc)

# Useful performance indicators for the industry
industry_perf = industry.copy()
industry_perf["revenue_per_employee"] = industry_perf["Выручка"] / industry_perf["Среднесписочная численность"]
industry_perf["profit_margin"] = industry_perf["Чистая прибыль"] / industry_perf["Выручка"]
industry_perf["assets_per_employee"] = industry_perf["Внеоборотные активы"] / industry_perf["Среднесписочная численность"]

perf_desc = industry_perf[[
    "revenue_per_employee",
    "profit_margin",
    "assets_per_employee",
]].describe().T
save_df(perf_desc, "industry_performance_ratios.csv")

# Cobb-Douglas production function:
# Output: revenue (strictly positive in the dataset)
# Inputs: cash, fixed assets, loans, materials, labor
# Rows with any nonpositive selected variable are dropped before logs.
prod_cols = [
    "Выручка",
    "Денежные средства",
    "Внеоборотные активы",
    "Кредиты и займы",
    "Остаток материалов",
    "Среднесписочная численность",
]

prod_raw, prod_log = safe_log_df(industry, prod_cols)

y = prod_log["Выручка"]
X = sm.add_constant(prod_log[[
    "Денежные средства",
    "Внеоборотные активы",
    "Кредиты и займы",
    "Остаток материалов",
    "Среднесписочная численность",
]])

# 3) OLS for Cobb-Douglas
ols_cd = sm.OLS(y, X).fit(cov_type="HC1")
cd_table = result_to_table(ols_cd)
save_df(cd_table, "industry_cobb_douglas_ols.csv")

returns_to_scale = ols_cd.params.drop("const").sum()
with open(os.path.join(OUTPUT_DIR, "industry_returns_to_scale.txt"), "w", encoding="utf-8") as f:
    f.write(f"Returns to scale (sum of elasticities): {returns_to_scale:.6f}\n")

print("\n=== Cobb-Douglas OLS ===")
print(ols_cd.summary())
print(f"\nReturns to scale (sum of elasticities): {returns_to_scale:.6f}")


# 5) Stochastic frontier for the same production function
def sfa_neg_loglik(params, y, X):
    beta = params[:-2]
    log_sigma_v, log_sigma_u = params[-2], params[-1]
    sigma_v = np.exp(log_sigma_v)
    sigma_u = np.exp(log_sigma_u)

    sigma = np.sqrt(sigma_v**2 + sigma_u**2)
    lam = sigma_u / sigma_v

    eps = y - X @ beta
    z = eps / sigma

    ll = np.log(2.0) - np.log(sigma) + norm.logpdf(z) + norm.logcdf(-lam * z)
    return -np.sum(ll)

# OLS starting values are usually a good initialization.
start_params = np.r_[ols_cd.params.values, np.log(np.std(ols_cd.resid) / np.sqrt(2)), np.log(np.std(ols_cd.resid) / np.sqrt(2))]
sfa_fit = minimize(
    sfa_neg_loglik,
    start_params,
    args=(y.values, X.values),
    method="L-BFGS-B",
)

if not sfa_fit.success:
    print("\nWARNING: SFA optimizer did not fully converge:", sfa_fit.message)

sfa_params = sfa_fit.x
beta_sfa = sfa_params[:-2]
sigma_v = np.exp(sfa_params[-2])
sigma_u = np.exp(sfa_params[-1])
sigma = np.sqrt(sigma_v**2 + sigma_u**2)
gamma = sigma_u**2 / (sigma_u**2 + sigma_v**2)

sfa_table = pd.DataFrame({
    "coef": np.r_[beta_sfa, sigma_v, sigma_u, gamma],
}, index=list(X.columns) + ["sigma_v", "sigma_u", "gamma"])
save_df(sfa_table, "industry_stochastic_frontier_parameters.csv")

# Technical efficiency scores
eps = y.values - X.values @ beta_sfa
mu_star = -(eps * sigma_u**2) / (sigma**2)
sigma_star = (sigma_u * sigma_v) / sigma

a = mu_star / sigma_star
mills_ratio = np.exp(norm.logpdf(a) - norm.logsf(a))
eu = mu_star + sigma_star * mills_ratio
te = np.exp(-eu)

te_df = pd.DataFrame({
    "technical_efficiency": te
}, index=prod_raw.index)
save_df(te_df, "industry_technical_efficiency.csv")

print("\n=== Stochastic frontier results ===")
print("Coefficients:")
for name, val in zip(list(X.columns), beta_sfa):
    print(f"  {name}: {val:.6f}")
print(f"  sigma_v: {sigma_v:.6f}")
print(f"  sigma_u: {sigma_u:.6f}")
print(f"  gamma:   {gamma:.6f}")
print(f"Mean technical efficiency: {np.mean(te):.6f}")

# Save a short note about Olley-Pakes limitations for this dataset.
with open(os.path.join(OUTPUT_DIR, "olley_pakes_note.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Olley-Pakes is not estimated here because the workbook is a cross-section, "
        "and the standard OP approach needs panel data and an investment proxy "
        "for unobserved productivity."
    )

### Примечание

Если в Excel-файле остался только один лист с названием `Данные`, ноутбук выполнится без дополнительных изменений.